# YOLO11-seg plate segmenter v1/v2 비교 검수

이 노트북은 Colab L4 GPU에서 기존 모델(v1)과 food_visible-only 샘플을 포함한 v2 모델을 같은 이미지 50장으로 비교합니다. 각 이미지마다 원본, 정답 라벨, v1 예측, v2 예측을 시각화하고, 클래스별 IoU/검출률을 계산해 어떤 모델을 운영에 쓸지 판단합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/final_1_team/apps/api/food-image-cleanup-pipeline')
assert PROJECT_ROOT.is_dir(), f'프로젝트 경로를 확인하세요: {PROJECT_ROOT}'
%cd {PROJECT_ROOT}

In [ ]:
!pip install --prefer-binary --upgrade-strategy only-if-needed -r requirements-colab.txt
import torch
print('CUDA 사용 가능:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'

## 비교 설정

- `MODEL_V1_WEIGHTS`: 기존 변환 방식으로 학습한 모델
- `MODEL_V2_WEIGHTS`: `food_visible` 단독 샘플까지 포함하고 고성능 설정으로 학습한 v2_hp 모델
- 기본 테스트셋은 v2 데이터셋입니다. v2가 food-only 케이스를 포함하므로 두 모델의 차이를 보기 좋습니다.

In [ ]:
import json, random, shutil
from pathlib import Path

MODEL_V1_WEIGHTS = Path('runs/plate_segmenter/yolo11n_plate_seg_reviewed_192/weights/best.pt')
MODEL_V2_WEIGHTS = Path('runs/plate_segmenter/yolo11s_plate_seg_reviewed_192_v2_hp/weights/best.pt')
DATASET_ROOT = Path('data/training/plate_segmentation/yolo_plate_segmentation_reviewed_192_v2')
SAMPLE_SIZE = 50
SEED = 42
CONF = 0.25
IMGSZ = 1024
DEVICE = 0
OUT_DIR = Path('runs/plate_segmenter_comparison/reviewed_192_v1_vs_v2')

assert MODEL_V1_WEIGHTS.is_file(), f'v1 가중치가 없습니다: {MODEL_V1_WEIGHTS}'
assert MODEL_V2_WEIGHTS.is_file(), f'v2 가중치가 없습니다: {MODEL_V2_WEIGHTS}'
assert DATASET_ROOT.is_dir(), f'데이터셋 폴더가 없습니다: {DATASET_ROOT}'
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

candidate_images = []
for split in ('test', 'val', 'train'):
    candidate_images.extend(sorted((DATASET_ROOT / 'images' / split).glob('*')))
candidate_images = [p for p in candidate_images if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}]
assert candidate_images, '비교할 이미지가 없습니다.'
random.Random(SEED).shuffle(candidate_images)
sample_images = candidate_images[:min(SAMPLE_SIZE, len(candidate_images))]
print('sample images:', len(sample_images))
print('v1:', MODEL_V1_WEIGHTS)
print('v2:', MODEL_V2_WEIGHTS)
print('dataset:', DATASET_ROOT)
print('out:', OUT_DIR)

In [ ]:
import cv2
import numpy as np
from PIL import Image
from ultralytics import YOLO

CLASS_NAMES = {0: 'plate_full', 1: 'food_visible'}
COLORS = {
    0: np.array([255, 140, 0], dtype=np.uint8),   # orange: plate_full
    1: np.array([0, 220, 120], dtype=np.uint8),   # green: food_visible
}

def label_path_for(image_path: Path) -> Path:
    split = image_path.parent.name
    return DATASET_ROOT / 'labels' / split / f'{image_path.stem}.txt'

def yolo_label_masks(label_path: Path, shape: tuple[int, int]) -> dict[int, np.ndarray]:
    h, w = shape
    masks = {0: np.zeros((h, w), dtype=np.uint8), 1: np.zeros((h, w), dtype=np.uint8)}
    if not label_path.is_file():
        return masks
    for line in label_path.read_text(encoding='utf-8').splitlines():
        parts = line.strip().split()
        if len(parts) < 7:
            continue
        cls = int(float(parts[0]))
        coords = np.array([float(v) for v in parts[1:]], dtype=np.float32).reshape(-1, 2)
        pts = np.column_stack((coords[:, 0] * w, coords[:, 1] * h)).round().astype(np.int32)
        if cls in masks and len(pts) >= 3:
            cv2.fillPoly(masks[cls], [pts], 255)
    return masks

def prediction_masks(model: YOLO, image_path: Path, shape: tuple[int, int]) -> dict[int, np.ndarray]:
    h, w = shape
    result = model.predict(source=str(image_path), imgsz=IMGSZ, conf=CONF, device=DEVICE, verbose=False)[0]
    masks = {0: np.zeros((h, w), dtype=np.uint8), 1: np.zeros((h, w), dtype=np.uint8)}
    if result.masks is None or result.boxes is None:
        return masks
    mask_data = result.masks.data.detach().cpu().numpy()
    classes = result.boxes.cls.detach().cpu().numpy().astype(int)
    for mask, cls in zip(mask_data, classes):
        if cls not in masks:
            continue
        binary = (mask >= 0.5).astype(np.uint8) * 255
        if binary.shape != (h, w):
            binary = cv2.resize(binary, (w, h), interpolation=cv2.INTER_NEAREST)
        masks[cls] = np.maximum(masks[cls], binary)
    return masks

def mask_iou(gt: np.ndarray, pred: np.ndarray) -> float | None:
    gt_bool = gt > 0
    pred_bool = pred > 0
    union = np.logical_or(gt_bool, pred_bool).sum()
    if union == 0:
        return None
    return float(np.logical_and(gt_bool, pred_bool).sum() / union)

def overlay_masks(image_rgb: np.ndarray, masks: dict[int, np.ndarray], alpha: float = 0.45) -> np.ndarray:
    out = image_rgb.copy()
    for cls, mask in masks.items():
        region = mask > 0
        if not np.any(region):
            continue
        color = COLORS[cls]
        out[region] = (out[region].astype(np.float32) * (1.0 - alpha) + color.astype(np.float32) * alpha).astype(np.uint8)
    return out

model_v1 = YOLO(str(MODEL_V1_WEIGHTS))
model_v2 = YOLO(str(MODEL_V2_WEIGHTS))
print('models loaded')

In [ ]:
from IPython.display import display, Image as DisplayImage
import matplotlib.pyplot as plt
from collections import defaultdict

rows = []
page_paths = []
for index, image_path in enumerate(sample_images, start=1):
    image_rgb = np.array(Image.open(image_path).convert('RGB'))
    h, w = image_rgb.shape[:2]
    gt = yolo_label_masks(label_path_for(image_path), (h, w))
    pred_v1 = prediction_masks(model_v1, image_path, (h, w))
    pred_v2 = prediction_masks(model_v2, image_path, (h, w))
    record = {'image': str(image_path), 'split': image_path.parent.name}
    for cls, name in CLASS_NAMES.items():
        record[f'{name}_gt_pixels'] = int(np.count_nonzero(gt[cls]))
        record[f'{name}_v1_pixels'] = int(np.count_nonzero(pred_v1[cls]))
        record[f'{name}_v2_pixels'] = int(np.count_nonzero(pred_v2[cls]))
        record[f'{name}_v1_iou'] = mask_iou(gt[cls], pred_v1[cls])
        record[f'{name}_v2_iou'] = mask_iou(gt[cls], pred_v2[cls])
    rows.append(record)

    fig, axes = plt.subplots(1, 4, figsize=(18, 5))
    panels = [
        ('original', image_rgb),
        ('ground truth', overlay_masks(image_rgb, gt)),
        ('v1 prediction', overlay_masks(image_rgb, pred_v1)),
        ('v2 prediction', overlay_masks(image_rgb, pred_v2)),
    ]
    for ax, (title, panel) in zip(axes, panels):
        ax.imshow(panel)
        ax.set_title(title)
        ax.axis('off')
    fig.suptitle(f'{index:02d}. {image_path.name} | orange=plate_full, green=food_visible')
    fig.tight_layout()
    page_path = OUT_DIR / f'comparison_{index:03d}_{image_path.stem}.jpg'
    fig.savefig(page_path, dpi=140)
    plt.close(fig)
    page_paths.append(page_path)

for page_path in page_paths[:12]:
    display(DisplayImage(filename=str(page_path)))
print('saved comparison pages:', len(page_paths), OUT_DIR)

In [ ]:
import csv, math

def mean_defined(values):
    values = [v for v in values if v is not None]
    return sum(values) / len(values) if values else None

def detection_rate(records, cls_name: str, model_tag: str) -> float:
    gt_key = f'{cls_name}_gt_pixels'
    pred_key = f'{cls_name}_{model_tag}_pixels'
    positives = [r for r in records if r[gt_key] > 0]
    if not positives:
        return 0.0
    return sum(1 for r in positives if r[pred_key] > 0) / len(positives)

summary = {}
for cls_name in ('plate_full', 'food_visible'):
    for model_tag in ('v1', 'v2'):
        summary[f'{cls_name}_{model_tag}_mean_iou'] = mean_defined([r[f'{cls_name}_{model_tag}_iou'] for r in rows])
        summary[f'{cls_name}_{model_tag}_detection_rate'] = detection_rate(rows, cls_name, model_tag)

food_only = [r for r in rows if r['plate_full_gt_pixels'] == 0 and r['food_visible_gt_pixels'] > 0]
summary['food_only_images'] = len(food_only)
for model_tag in ('v1', 'v2'):
    summary[f'food_only_{model_tag}_food_visible_mean_iou'] = mean_defined([r[f'food_visible_{model_tag}_iou'] for r in food_only])
    summary[f'food_only_{model_tag}_plate_false_positive_rate'] = (
        sum(1 for r in food_only if r[f'plate_full_{model_tag}_pixels'] > 0) / len(food_only)
        if food_only else None
    )

csv_path = OUT_DIR / 'per_image_metrics.csv'
with csv_path.open('w', encoding='utf-8-sig', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)
summary_path = OUT_DIR / 'summary_metrics.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('csv:', csv_path)
print('summary:', summary_path)

## 모델 선택 기준

v2를 선택합니다.

- `food_visible_v2_mean_iou`가 v1보다 같거나 높다.
- `food_visible_v2_detection_rate`가 v1보다 같거나 높다.
- `food_only_v2_food_visible_mean_iou`가 v1보다 높다.
- `food_only_v2_plate_false_positive_rate`가 v1보다 낮거나 같다.
- 시각 비교에서 food-only 이미지에 plate_full 가짜 예측이 적고, 음식 외곽이 더 안정적이다.

v1을 선택합니다.

- `plate_full_v2_mean_iou`가 v1보다 0.05 이상 낮다.
- preserve_original_plate 결과에서 접시 외곽 누락이 v2에서 더 자주 보인다.
- food-only 개선보다 plate_full 품질 저하가 운영 리스크로 더 크다.

운영 반영은 선택한 모델의 `best.pt`를 `models/yolo11n_plate_seg.pt`로 복사한 뒤 `configs/pipeline.yaml`의 `plate_segmenter.enabled`를 `true`로 설정합니다.